In [ ]:
import pandas as pd
import json

from dataclasses import dataclass, field, asdict

from pathlib import Path

ROOT = Path.cwd().parents[1]

DATASET_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

CSV_DIR = RESULTS_DIR / "csvs" / "data"

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
print(DATASET_DIR.exists())

In [ ]:
@dataclass
class BoundingBox:
    category: int   #Fire-1, Smoke-0
    coordinates: tuple[float, float, float, float] #x_center, y_center, width, height

@dataclass
class LabelRecord:
    path: str
    split: str
    boxes: list[BoundingBox]
    label: int = field(init=False) # contains fire/smoke-1, doesnt-0

    def __post_init__(self):
        self.label = int(bool(self.boxes))

In [ ]:
records: list[LabelRecord] = []
raw_records: list[LabelRecord] = []

invalid_count = 0
for split in DATASET_DIR.iterdir():
    if not split.is_dir():
        continue

    print(split.name)

    # Uses rglob as train has a lower level dir level so it recursively finds it
    for label_path in (split / "labels").rglob("*.txt"):
        boxes: list[BoundingBox] = []
        raw_boxes: list[BoundingBox] = []

        parts = list(label_path.parts)
        parts[parts.index("labels")] = "images"

        img_path = Path(*parts).with_suffix(".jpg")

        if not img_path.exists():
            print("[ERROR] Image does not exist in dir:", img_path)
            continue
        
        with open(label_path) as f:
            contents = f.read()

        if contents.strip():
            for lines in contents.splitlines():
                parts = lines.split()

                coordinates = tuple(float(x) for x in parts[1:])

                box = BoundingBox(
                        category=int(parts[0]),
                        coordinates=coordinates
                    )

                raw_boxes.append(box)

                # Some boxes can have width or heigh 0 so remove them as they're invalid
                if coordinates[2] <= 0 or coordinates[3] <= 0:
                    invalid_count += 1
                    continue
                
                boxes.append(box)       

        records.append(
            LabelRecord(
                path=img_path.relative_to(DATASET_DIR).as_posix(),
                split=split.name,
                boxes=boxes
            )
        )

        raw_records.append(
            LabelRecord(
                path=img_path.relative_to(DATASET_DIR).as_posix(),
                split=split.name,
                boxes=raw_boxes
            )
        )

In [ ]:
invalid_count

In [ ]:
raw_df = pd.DataFrame([asdict(record) for record in raw_records])
df = pd.DataFrame([asdict(record) for record in records])

# Converts boxes from string to valid json for saving
raw_df["boxes"] = raw_df["boxes"].apply(json.dumps)
df["boxes"] = df["boxes"].apply(json.dumps)

df

In [ ]:
raw_df.to_csv(CSV_DIR / "metadata_raw.csv", index=False)
df.to_csv(CSV_DIR / "metadata.csv", index=False)

print(f"Processed {len(df)} images")